<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/06/Module6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic Training Data

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [ ]:
NUM_TRAIN_EXAMPLES = 100  # @param {type:"number"}
NUM_VAL_EXAMPLES = 50  # @param {type:"number"}
NUM_TEST_EXAMPLES = 10 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}

DATA_FOLDER = "./data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5-nano"

# Create it if it doesn't exist
#os.makedirs(DATA_FOLDER, exist_ok=True)

# Check if it exists
#print("Folder exists?", os.path.exists(DATA_FOLDER))
#print("Full path:", os.path.abspath(DATA_FOLDER))

## Dataset diversity

In [ ]:
TOPICS= [
    # Child Topics
    "silly_animals",
    "animal_sounds",
    "funny_foods",
    "animals_common",
    "colors_shapes",
    "weather_basic",
    "toys",
    "school_objects",
    "food_simple",
    "hiding_spots",
    "dress_up_costumes",
    "magic_tricks_simple",
    "pirates_treasure",
    "dinosaurs",
    "space_aliens",
    "superheroes_powers",
    "monsters_not_scary",
    "fairy_tale_mashups",
    # Teen Topics
    "school_memes",
    "weird_rumors",
    "awkward_moments",
    "friend_group_chaos",
    "pranks_harmless",
    "gaming_items_powerups",
    "boss_fight_logic",
    "plot_twists",
    "mystery_clues",
    "secret_identities",
    "robot_glitches",
    "time_loop_scenarios",
    "dream_weirdness",
    "choose_your_own_adventure",
    "texting_misreads",
    "autocorrect_mischief",
    "music_earworms",
    "sports_trick_plays",
    "fantasy_creatures",
    "urban_legends_light",
    # Adult Topics
    "bad_puns",
    "dad_jokes",
    "trick_questions",
    "misdirection_riddles",
    "double_meanings",
    "idioms_literalized",
    "kitchen_catastrophes",
    "pet_shenanigans",
    "travel_mishaps",
    "party_games",
    "board_game_brain_teasers",
    "office_absurdities",
    "mystery_noir_parodies",
    "fairy_tales_reimagined",
    "sci_fi_paradoxes_light",
    "crime_scene_silly",
    "impossible_inventions",
    "tiny_annoyances",
    "secret_rules_of_life",
    "who_am_i_charades"
]

DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard",
    "trick"
]

DIFFICULTY_LEVEL_WEIGHTS = [0.4, 0.4, 0.1, 0.1]

RIDDLE_LENGTH = [
    "short",
    "medium",
    "long"
]

RIDDLE_LENGTH_WEIGHTS = [0.6, 0.3, 0.1]

CORRECTNESS = [
  "a correct answer to the riddle",
  "an incorrect answer to the riddle"
]
CORRECTNESS_WEIGHTS = [0.9, 0.1]


## Model for structured output

In [ ]:
from pydantic import BaseModel

class RiddleExample (BaseModel):
    riddle: str
    answer: str
    correct: bool

## Get OpenRouter API key

In [ ]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Conversation generation functions

In [ ]:
from httpx._transports.default import A
import openai
import os
from dataclasses import dataclass

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> RiddleExample | None:
    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=RiddleExample
    )

    return response.output_parsed

def map_riddle_length(riddle_length: str, topic: str) -> str:
    if riddle_length == "short":
        return f"a short riddle (1 sentences) about {topic}"
    elif riddle_length == "medium":
        return f"a medium-style riddle (2 sentences) about {topic}"
    elif riddle_length == "long":
        return f"a short story-style riddle (3 sentences) about {topic}"
    else:
        return f"a riddle about {topic}"

def create_conversation(
    topic: str,
    correctness: str,
    riddle_length: str) -> RiddleExample | None:

    request = map_riddle_length(riddle_length, topic)

    prompt = f"""
    You are putting together riddles with random answers.

    Generate {request} and a {correctness}.

    Return the following:

    - The riddle
    - The proposed answer to the riddle
    - True/False indicating whether or not the answer is correct

    """

    return generate_completion(prompt)

## Dataset generation functions

In [ ]:
import random
import json
from tqdm import tqdm

def generate_dataset(num_examples: int, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):

          topic = random.choice(TOPICS)

          correctness = random.choices(
              CORRECTNESS,
              weights=CORRECTNESS_WEIGHTS
          )[0]

          riddle_length = random.choices(
              RIDDLE_LENGTH,
              weights=RIDDLE_LENGTH_WEIGHTS
          )[0]

          # Generate riddle
          conversation = None
          while conversation is None:
              conversation = create_conversation(
                  topic=topic,
                  correctness = correctness,
                  riddle_length = riddle_length
              )

          # Structured training template
          template = {
          "messages": [
              {
                  "role": "user",
                  "content": "Tell me a riddle."
              },
              {
                  "role": "assistant",
                  "content": conversation.riddle
              },
              {
                  "role": "user",
                  "content": conversation.answer
              },
              {
                  "role": "assistant",
                  "content": "Correct" if conversation.correct else "Incorrect"
              }
              ]
          }

          line = json.dumps(template, ensure_ascii=False) + "\n"
          f.write(line)
          f.flush()

        f.flush()
        f.close()

## Generate all the data!

In [ ]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
#generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
#generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)


100%|██████████| 100/100 [37:05<00:00, 22.25s/it]
